# D1-02 Brightway architecture and data model

This notebook makes the core Brightway abstractions visible before we import a real database later in Day 1.

We work with a tiny toy system so that projects, databases, activities, exchanges, and methods are easy to inspect in code.


## Learning goals

- Understand how `brightway` organizes work into projects.
- See how databases, activities, exchanges, and methods relate to each other.
- Create and inspect a tiny toy database.
- Practice basic `bw2data` inspection patterns before importing BAFU in `D1-04`.


## Background reference

- Mutel, C. (2017). *Brightway: An open source framework for life cycle assessment*. Journal of Open Source Software, 2(12), 236. https://doi.org/10.21105/joss.00236


## 1) Projects are the top-level container

A project contains databases, LCIA methods, and related metadata.
Objects from one project do not interact with objects in another project.


In [ ]:
import bw2data as bd

project_name = 'barcelona-architecture-demo'
bd.projects.set_current(project_name)

print('Current project:', bd.projects.current)
print('Number of known projects on this machine:', len(list(bd.projects)))


## 2) What is inside a project?

A project can contain many databases and many LCIA methods.
Before creating our toy system, inspect the project itself.


In [ ]:
print('Project directory:', bd.projects.dir)
print('Databases currently available:', list(bd.databases))
print('Number of methods currently available:', len(bd.methods))


## 3) Build a tiny toy system

We create:

- one toy biosphere database
- one toy technosphere database with two activities
- one toy LCIA method

The example is deliberately small so that the architecture stays visible.


In [ ]:
biosphere_name = 'biosphere_toy_arch'
tech_name = 'tech_toy_arch'
method_key = ('Toy method', 'climate change', 'toy GWP')

if biosphere_name not in bd.databases:
    bd.Database(biosphere_name).write({
        (biosphere_name, 'co2'): {
            'name': 'Carbon dioxide, fossil',
            'unit': 'kilogram',
            'type': 'emission',
        }
    })

if tech_name not in bd.databases:
    bd.Database(tech_name).write({
        (tech_name, 'A'): {
            'name': 'Activity A',
            'unit': 'kilogram',
            'location': 'GLO',
            'exchanges': [
                {'input': (tech_name, 'A'), 'amount': 1.0, 'type': 'production'},
                {'input': (tech_name, 'B'), 'amount': 0.4, 'type': 'technosphere'},
                {'input': (biosphere_name, 'co2'), 'amount': 1.2, 'type': 'biosphere'},
            ],
        },
        (tech_name, 'B'): {
            'name': 'Activity B',
            'unit': 'kilogram',
            'location': 'GLO',
            'exchanges': [
                {'input': (tech_name, 'B'), 'amount': 1.0, 'type': 'production'},
                {'input': (biosphere_name, 'co2'), 'amount': 0.3, 'type': 'biosphere'},
            ],
        },
    })

if method_key not in bd.methods:
    bd.Method(method_key).register(unit='kilogram CO2-eq')
    bd.Method(method_key).write([((biosphere_name, 'co2'), 1.0)])

print('Databases:', list(bd.databases))
print('Toy method available:', method_key in bd.methods)


## Checkpoint 1

Create a temporary project called `barcelona-rlcia-sandbox`, confirm that it became the current project,
then switch back to `barcelona-architecture-demo`.


In [ ]:
# TODO
# Hint: use bd.projects.set_current(...)


In [ ]:
sandbox_name = 'barcelona-rlcia-sandbox'
bd.projects.set_current(sandbox_name)
print('Sandbox project:', bd.projects.current)

bd.projects.set_current(project_name)
print('Back to architecture demo project:', bd.projects.current)

# Optional cleanup after the course:
# bd.projects.delete_project(sandbox_name, delete_dir=True)


## 4) A database contains activities

Activities are the objects we query when we build product systems.


In [ ]:
db = bd.Database(tech_name)
print('Database object:', db)
print('Number of activities:', len(db))

for act in db:
    print('-', act['name'], '| key =', act.key, '| location =', act.get('location'))


## 5) An activity contains exchanges

Exchanges link an activity to:

- itself, through a production exchange
- another activity, through a technosphere exchange
- an elementary flow, through a biosphere exchange


In [ ]:
activity_a = bd.get_activity((tech_name, 'A'))

print('Selected activity:', activity_a['name'])
print('Unit:', activity_a['unit'])
print('Location:', activity_a.get('location'))

print('Production exchanges:')
for exc in activity_a.production():
    print('  -', exc.input['name'], '| amount =', exc['amount'])

print('Technosphere exchanges:')
for exc in activity_a.technosphere():
    print('  -', exc.input['name'], '| amount =', exc['amount'])

print('Biosphere exchanges:')
for exc in activity_a.biosphere():
    print('  -', exc.input['name'], '| amount =', exc['amount'])


## Checkpoint 2

Store one technosphere exchange from `activity_a` in `tech_exc` and one biosphere exchange in `bio_exc`.
Print the input name and amount for both.


In [ ]:
# TODO
# tech_exc = ...
# bio_exc = ...


In [ ]:
tech_exc = list(activity_a.technosphere())[0]
bio_exc = list(activity_a.biosphere())[0]

print('Technosphere input:', tech_exc.input['name'], '| amount =', tech_exc['amount'])
print('Biosphere input:', bio_exc.input['name'], '| amount =', bio_exc['amount'])


## 6) Methods connect elementary flows to impacts

An LCIA method stores characterization factors for biosphere flows.


In [ ]:
method = bd.Method(method_key)
print('Method metadata:', method.metadata)
print('Method CFs:', method.load())

bio_flow = bd.get_activity((biosphere_name, 'co2'))
print('Elementary flow used in the toy method:', bio_flow['name'])


## Common mistake

Switching projects changes what databases and methods are visible.
If something you created seems to have disappeared, first check the current project.


## Recap

After this notebook, you should be able to:

- explain what a project contains
- create or switch projects
- recognize that databases contain activities
- inspect production, technosphere, and biosphere exchanges
- explain that methods characterize biosphere flows, not technosphere flows
